# Deploy a DeepSeek-V4 models on Amazon SageMaker AI Endpoint

## DeepSeek-V4: Towards Highly Efficient Million-Token Context Intelligence

📄 [Technical Report](https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro)

### Introduction

Here is a preview version of DeepSeek-V4 series, including two strong Mixture-of-Experts (MoE) language models — **DeepSeek-V4-Pro** with 1.6T parameters (49B activated) and **DeepSeek-V4-Flash** with 284B parameters (13B activated) — both supporting a context length of **one million tokens**.

DeepSeek-V4 series incorporate several key upgrades in architecture and optimization:

- **Hybrid Attention Architecture:** We design a hybrid attention mechanism combining Compressed Sparse Attention (CSA) and Heavily Compressed Attention (HCA) to dramatically improve long-context efficiency. In the 1M-token context setting, DeepSeek-V4-Pro requires only **27% of single-token inference FLOPs** and **10% of KV cache** compared with DeepSeek-V3.2.
- **Manifold-Constrained Hyper-Connections (mHC):** We incorporate mHC to strengthen conventional residual connections, enhancing stability of signal propagation across layers while preserving model expressivity.
- **Muon Optimizer:** We employ the Muon optimizer for faster convergence and greater training stability.

We pre-train both models on more than **32T diverse and high-quality tokens**, followed by a comprehensive post-training pipeline. The post-training features a two-stage paradigm: independent cultivation of domain-specific experts (through SFT and RL with GRPO), followed by unified model consolidation via on-policy distillation, integrating distinct proficiencies across diverse domains into a single model.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [3]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves 'Creating' or "Updating" state."""
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## Container

In [40]:
instance = {"type": "ml.p5en.48xlarge", "num_gpu": 8}
model_id = "deepseek-ai/DeepSeek-V4-Flash"
#model_id = "deepseek-ai/DeepSeek-V4-Pro"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 3600
variant_name = "v1"

##### Amazon SageMaker AI makes model deployment process very easy. The deployment steps are exactly the same regardless of model framework container you choose. Below we show you 2 options:
1. DLC vLLM (0.20.1)
2. DLC SGLang (BYOC version)

#### Please chose one option and run only the cell for your container of choice

#### We will be using native DeepSeek v4 multi-token predictions (MTP). The net effect is higher throughput (more tokens per second) with no degradation in output quality, since the verification step ensures the final output matches what standard autoregressive decoding would produce. It's essentially "free" speculation baked into the model itself, removing the overhead of training and serving a separate smaller draft model.

### Option 1: vLLM

The configuration below can be used to deploy both `Flash` and `Pro` DeepSeen v4 models.

Also, the configuration below is latency optimized (only `SM_VLLM_TENSOR_PARALLEL_SIZE` is set). 

To deploy "balanced" configuration - make sure to set:
- `SM_VLLM_TENSOR_PARALLEL_SIZE`
- `SM_VLLM_ENABLE_EXPERT_PARALLEL`

To deploy "throughput" optimized configuration:
- comment out `SM_VLLM_TENSOR_PARALLEL_SIZE`
- set `SM_VLLM_ENABLE_EXPERT_PARALLEL`
- set `SM_VLLM_DATA_PARALLEL_SIZE`



In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:vllm:0.20.1-gpu-py312-cu130-ubuntu22.04-sagemaker"

spec_config = {"method": "mtp", "num_speculative_tokens": 3}

compilation_config = {"mode": 0, "cudagraph_mode": "FULL_DECODE_ONLY"}

common_env = {
    "HF_TOKEN": "<YOUR_TOKEN_HERE>",
    "SM_NUM_GPUS": json.dumps(instance["num_gpu"]),
    # Uncomment for Pro model
    #"VLLM_ENGINE_READY_TIMEOUT_S": "3600",
    #"VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "0",
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    #"SM_VLLM_ENABLE_EXPERT_PARALLEL": "true",
    #"SM_VLLM_DATA_PARALLEL_SIZE": "4",
    "SM_VLLM_MAX_MODEL_LEN": "65536",
    "SM_VLLM_NO_ENABLE_FLASHINFER_AUTOTUNE": "true",
    "SM_VLLM_TRUST_REMOTE_CODE": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_BLOCK_SIZE": "256",
    "SM_VLLM_TOKENIZER_MODE": "deepseek_v4",
    "SM_VLLM_TOOL_CALL_PARSER": "deepseek_v4",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_REASONING_PARSER": "deepseek_v4",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps(spec_config),
    # Uncomment for Pro model
    #"SM_VLLM_GPU_MEMORY_UTILIZATION": "0.95",
    #"SM_VLLM_MAX_NUM_SEQS": "512",
    #"SM_VLLM_MAX_NUM_BATCHED_TOKENS": "512",
    #"SM_VLLM_COMPILATION_CONFIG": json.dumps(compilation_config),
}
env = common_env | vllm_env

reasoning_keyword = "reasoning"

### Option 2: SGLang

At the time of publishing (05/05/2026), SGLang requires a purposely build container to deploy DeepSeek v4 models.

Please refer to SGLang [documentation](https://docs.sglang.io/cookbook/autoregressive/DeepSeek/DeepSeek-V4) and to the example [notebook](https://github.com/aws-samples/sagemaker-genai-hosting-examples/tree/main/02-engines/SGLang) for additional instructions.

In [ ]:
#inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/sglang:0.5.10-gpu-py312-cu129-ubuntu24.04-sagemaker"
inference_image = f"<YOUR_BYOC_SGLANG_IMAGE_HERE>"

deepep_config = {"normal_dispatch":{"num_sms":96},"normal_combine":{"num_sms":96}}

common_env = {
    "SGLANG_DSV4_FP4_EXPERTS": "1",
    "SGLANG_JIT_DEEPGEMM_PRECOMPILE": "0"
}
sgl_env = {
    "SM_SGLANG_MODEL_PATH": model_id,
    "SM_SGLANG_TRUST_REMOTE_CODE": "true",
    "SM_SGLANG_CUDA_GRAPH_MAX_BS": "128",
    "SM_SGLANG_MAX_RUNNING_REQUESTS": "256",
    "SM_SGLANG_TP": "8",
    "SM_SGLANG_CONTEXT_LENGTH": "32768",
    "SM_SGLANG_MOE_RUNNER_BACKEND": "marlin",
    "SM_SGLANG_TOOL_CALL_PARSER": "deepseekv4",
    "SM_SGLANG_REASONING_PARSER": "deepseek-v4",
    "SM_SGLANG_SPECULATIVE_ALGO": "EAGLE",
    "SM_SGLANG_SPECULATIVE_NUM_STEPS": "3",
    "SM_SGLANG_SPECULATIVE_EAGLE_TOPK": "1",
    "SM_SGLANG_SPECULATIVE_NUM_DRAFT_TOKENS": "4",
}
env = common_env | sgl_env

reasoning_keyword = "reasoning_content"

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
capacity_reservation = {
    'CapacityReservationPreference': 'capacity-reservations-only',
    'MlReservationArn': '<YOUR_FTP_ARN_HERE>'  # replace with you FTP ARN
}

_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            # Uncomment if you have capacity reservation that can be used for the deployment
            #"CapacityReservationConfig": capacity_reservation,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

## Inference examples

### Text inference

In [45]:
payload = {
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))

✅ Response time: 1.40s



### Content:
---

Hi there! 👋 I'm DeepSeek, an AI assistant created by DeepSeek (深度求索). I'm here to help you with questions, conversations, problem-solving, and creative tasks!

Here are a few things about me:
- 🆓 **Completely free** to use - no charges, no hidden fees
- 📚 **1M context window** - I can process huge amounts of text at once (like entire book trilogies!)
- 🔗 **File upload support** - I can read text from images, PDFs, Word docs, Excel files, PowerPoint presentations, and more
- 🌐 **Web search capability** - though you'll need to manually enable it in the app
- 📱 **Available on mobile** with voice input support
- 🗓️ **Knowledge cutoff**: May 2025

I'm designed to be warm, helpful, and detailed in my responses. Whether you need help with homework, writing, analysis, coding, or just want to have a thoughtful conversation - I'm here for you!

What can I help you with today? 😊

### Text generation, no reasoning

In [46]:
payload = {
    "messages": [
        {"role": "user", "content": "What is the derivative of x^3 * ln(x)?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))

✅ Response time: 1.34s



### Content:
---

We are looking for the derivative of:

\[
f(x) = x^3 \cdot \ln(x)
\]

This is a product of two functions:

- \( u = x^3 \)  
- \( v = \ln(x) \)

Using the product rule:

\[
f'(x) = u' v + u v'
\]

First, compute the derivatives:

- \( u' = 3x^2 \)  
- \( v' = \frac{1}{x} \)

Now apply the product rule:

\[
f'(x) = (3x^2) \cdot \ln(x) + (x^3) \cdot \frac{1}{x}
\]

Simplify the second term:

\[
x^3 \cdot \frac{1}{x} = x^2
\]

So we get:

\[
f'(x) = 3x^2 \ln(x) + x^2
\]

Factor out \( x^2 \):

\[
\boxed{x^2 (3\ln(x) + 1)}
\]

### Text inference with reasoning

In [47]:
payload = {
    "messages": [
        {"role": "user", "content": "Analyze why speculative decoding works better for long-context language models."}
    ],
    "chat_template_kwargs": {
        "thinking": True,
        "reasoning_effort": "high", # Can also use "max"
    },
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))

✅ Response time: 19.28s



### Reasoning:
---

Hmm, this is a technical question about speculative decoding and its effectiveness specifically for long-context language models. The user wants an analysis, not just a definition. I need to explain the core concept of speculative decoding first—it's a method to accelerate LLM inference by using a smaller, faster draft model to predict tokens that are then verified by the large target model in parallel. 

The key is to connect why this technique becomes particularly advantageous when the context is very long. Long-context models (e.g., those handling 128k or 1M tokens) face a severe bottleneck: the self-attention mechanism has O(n²) complexity in compute and O(n) in memory, making each forward pass very slow. Speculative decoding improves on this in two main ways.

First, it reduces the number of expensive target model calls. Instead of generating one token per forward pass, speculative decoding generates a block of tokens with one verification pass. For long contexts, where each target model forward pass is extremely costly due to the massive KV cache and attention computation, reducing the number of passes yields a significant wall-clock speedupasi.

Second, there's a caching and KV reuse benefit. In long-context scenarios, altering the context (like in editing tasks) is more common. Speculative decoding allows the target model's KV cache to be computed once for the long prefixable, and the draft model can propose tokens using a partial KV cache. The verification process can often reuse the existing cache for the shared prefix, whereas generating token-by-token would require updating the cache incrementally, which is slower for long contexts due to memory bandwidth constraints.

Third, the acceptance rate of draft tokens might be slightly higher in long contexts? Actually, it depends on the task. But the bigger win is that even a modest acceptance rate leads to a large absolute time savings because each draft model step is much faster than a target model step. The ratio of speed between draft and target model increases as context length grows (since target model's attention cost grows linearly with context, while draft model's smaller size means its cost grows less steeply). So the effective speedup (proposed tokens per verification) is amplified.

I should also mention speculative decoding's synergy with hardware parallelism: the verification step computes logits for all proposed tokens in one batch, which utilizes GPU parallelism better than sequential generation, and for long contexts, the memory bandwidth bottleneck is more pronounced, so batching multiple predictions amortizes the cost of loading the KV cache.

Potential counterpoints: The draft model must be good enough; if it's too poor, the acceptance rate drops, making the overhead not worth it. But for many long-context tasks (like summarization, document QA), the content is repetitive or has predictable patterns, so a simple draft model works well.

I'll structure the analysis: start with what speculative decoding is, then explain each reason specifically tied to long contexts: (1) Reduction in expensive forward passes due to increased cost per pass, (2) KV cache reuse benefits, (3) Improved parallelism and memory bandwidth efficiency, (4) Potential for higher acceptance rates in certain long-context tasks. Conclude with a summary linking these factors to the overall speedup principle.

Need to ensure the explanation is clear but not overly simplistic, as the user seems technically inclined. Use examples like Transformer inference cost formula to illustrate the time savings. Avoid unnecessary details about exact algorithm variants unless relevant.

Let me write a comprehensive yet concise analysis.

### Content:
---

This is an excellent question. Speculative decoding is a powerful inference acceleration technique, and its advantages become particularly pronounced for long-context language models. Let's break down the "why" by first understanding the core mechanism, then analyzing how long contexts amplify its benefits.

### The Core Idea of Speculative Decoding

Speculative decoding accelerates autoregressive generation (where you produce one token at a time) by using a **small, fast draft model** to propose a sequence of *K* tokens. The large, accurate **target model** (the one you actually want) then *verifies* all *K* tokens in a single, parallelizable forward pass.

**The Crucial Benefit**: Instead of *K* sequential forward passes of the slow target model, you only do 1 sequential pass of the fast draft model + 1 parallel verification pass of the target model. If the draft model is good, you generate *K* tokens with the "correctness" of the target model, but at a fraction of the latency.

Now, let's see why this is a game-changer for long contexts.

---

### Why Speculative Decoding Shines for Long-Context Models

The performance of the slow, target model degrades significantly with context length, while the draft model's degradation is much less severe. Speculative decoding directly exploits this asymmetry. The key factors are:

#### 1. The Quadratic (or Sub-Quadratic) Cost of Attention

The most computationally expensive part of a Transformer-based LLM forward pass is the attention mechanism.

- **The Bottleneck:** For a given context length `L`, the attention operation has a time and memory complexity that is at least **O(L)** (for linear/flash attention) and often **O(L²)** (for standard causal attention). This is not an issue for short contexts (e.g., 2k tokens), but becomes a massive overhead for long contexts (e.g., 128k, 1M tokens).
- **The Target Model's Pain Point:** Every single forward pass of the target model pays this high `O(L)` or `O(L²)` cost. Generating 10 tokens sequentially means paying this high cost 10 times.
- **The Draft Model's Advantage:** The draft model is typically much smaller (e.g., a 125M parameter model vs. a 7B or 70B target model), and often operates on a **truncated context** or uses a simpler architecture (e.g., an n-gram model or a tiny Transformer). Its attention cost for a forward pass is drastically lower.
- **The Amplified Leverage:** Speculative decoding replaces *K* expensive target model forward passes with just 1. The longer the context `L`, the more expensive each of those *K* target passes becomes, and therefore the more time you save by reducing them to a single pass">. *This is the primary reason.* The savings grow linearly (or worse) with context length.

**Example:** For a 100k token context:
- Naïve Generation: 10 target model passes, each working on a 100k-token context.
- Speculative Decoding: 10 draft model passes (much cheaper) + 1 target model pass on a 100k-token context.
You save the cost of 9 expensive 100k-token passes.

In contrast, for a 1k token context:
- The target model passes are already cheap)Skip. While you still save passes, the absolute time saved is much smaller and may not justify the overhead of the draft model and synchronization.

#### 2. Cache and Memory Bandwidth Bottlenecks

Large context sizes strain the **Key-Value (KV) cache**, which is a tensor used to store previous keys and values to avoid recomputing attention.

- **KV Cache Size:** The KV cache grows linearly with context length. For a 70B model at 100k tokens, the cache can be multiple gigabytes, often exceeding the memory of a single GPU accelerator.
- **Memory-Bound Operations:** The verification forward pass of the target model now needs to process this massive, multi-GB KV cache. Reading this from memory (HBM) is a major bottleneck, making each forward pass slow and memory-bandwidth-bound.
- **The Benefit of Fewer Passes:** Speculative decoding drastically reduces the number of times this massive KV cache needs to be loaded and processed by the expensive target model. The draft model, with its smaller KV cache (or no cache at all) is much less affected.
- **KV Cache Reuse:** Crucially, the verification step reuses the *same KV cache* from the prompt. The draft model's proposed tokens are verified against this existing, huge cache. The target model doesn't need to recompute the entire history for every draft token, only the new ones. This is a massive efficiency gain.

#### 3. The "Truncated Context" Advantage for the Draft Model

A highly effective strategy for speculative decoding in long contexts is to let the **draft model see only a recent slice of the context** (e.g., the last 2k tokens), while the target model sees the full 100k tokens.

- **Why this works:** The local structure of language (e.g., recent 2k tokens) is often sufficient for a good draft. The need for the full context emerges from complex, long-range reasoning tasks (e.g., summarization, multi-hop QA). The draft model can still predict plausible local sequences.
- **The Impact:** The draft model's forward passes become **extremely cheap** and context-independent. Its cost doesn't scale with `L`. This further magnifies the speedup, as the draft model's latency is constant while the target model's latency scales with `L`.

#### 4. Better Acceptance Rates in Long-Context Tasks

One might worry that a draft model using a truncated context would have a very low acceptance rate on the full-context target model, ruining the efficiency of speculative decoding. However, the opposite is often true for many long-context tasks.

- **Predictable Structure:** Long-context tasks often involve processing highly structured data (e.g., a long source document for summarization, a multi-turn conversation, an entire codebase). The structure and style of the output (e.g., a summary, a response, a code edit) are heavily constrained by the recent context and the task itself.
- **Local Coherence, Global Correctness:** The draft model can easily propose locally coherent text (e.g., "The answer is based on the fact that..."). The target model's job is to verify that this local coherence aligns with the global, long-range context (e.g., "...which was stated on page 45 of the document"). The acceptance rate for such plausible local sequences can remain high.
- **The "Free Lunch" of the Verification Pass:** Even if the acceptance rate is only 50%, a standard speculative decoding setup with *K=5* will generate, on average, 2.5 tokens per verification pass. This is still a net win, and the win is much larger because each verification pass is so expensive due to the long context.

#### 5. Synergy with Context-Efficient Architectures

Speculative decoding has a fantastic synergy with architectures designed for long contexts, such as **Linear Attention**, **FlashAttention**, and **State Space Models (e.g., Mamba, RWKV)**.

- **FlashAttention/Linear Attention:** These methods make the attention cost *O(L)* instead of *O(L²)*. This is great, but the target model's cost still scales linearly with `L`. Speculative decoding still reduces the *number* of these `O(L)` passes from *K* to 1.
- **State Space Models (SSMs):** SSMs like Mamba have a constant-time, state-based inference, which is fantastic for efficiency. However, they are challenging for speculative decoding. The draft model cannot easily propose tokens because the "draft" and "target" models are architecturally different (Transformer vs. SSM). But newer research is successfully combining SSM-based drafters with Transformer-based target models for long contexts, leveraging the SSM's speed for the draft and the Transformer's accuracy for verification.

### Summary Table

| Factor | Short Context (e.g., 2k tokens) | Long Context (e.g., 100k tokens) | Why Speculative Decoding Helps More for Long Context |
| :--- | :--- | :--- | :--- |
| **Cost of Target Forward Pass** | Low, negligible | Very High, often millions of FLOPs | Replacing *K* expensive passes with 1 yields massive absolute time savings. |
| **KV Cache Size** | Small, fits in fast memory | Large, memory-bandwidth bottleneck | Reduces the number of times this massive cache is loaded from slow memory. |
| **Draft Model Cost** | Comparable to target | Much, much cheaper | The draft model's cost doesn't scale with `L` (especially with truncation), creating a huge asymmetry. |
| **Acceptance Rate Risk** | Higher risk of mismatch | Often acceptable for structured tasks | The draft model can exploit local structure successfully, even when the target model uses long-range context. |
| **System Overhead** | Overhead can outweigh gains | Overhead is negligible | The high cost per target pass easily dwarfs the communication and draft model overhead. |

### Conclusion

Speculative decoding works better for long-context models because it directly attacks the **central bottleneck of long-context inference**: the escalating cost of each target model forward pass.

By replacing *K* expensive, memory-bandwidth-limited passes of the large model with *K* cheap passes of a small, context-agnostic draft model plus a single verification pass, speculative decoding achieves a **multiplicative speedup that increases with context length**. The longer the context, the more you save per rejected/ accepted token, making this technique a crucial enabler for deploying and using powerful long-context LLMs in practice.

### Streaming example with reasoning enabled

In [48]:
#
# Helper function for streamin invocation
#
import io
import json
import time
import boto3
from IPython.display import clear_output

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                print("Unknown event type:" + chunk)
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def stream_response(endpoint_name, inputs):
    body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": inputs}]}
        ],
        "stream": True,
        "chat_template_kwargs": {
            "thinking": True,
            "reasoning_effort": "max", # Can also use "high"
        },
    }

    resp = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        Body=json.dumps(body),
        ContentType="application/json",
    )

    event_stream = resp["Body"]
    start_json = b"{"
    full_response = ""
    start_time = time.time()
    reasoning_token_count = 0
    content_token_count = 0

    for line in LineIterator(event_stream):
        if line != b"" and start_json in line:
            data = json.loads(line[line.find(start_json):].decode("utf-8"))
            delta = data['choices'][0]['delta']
            reasoning = delta.get(reasoning_keyword, '')
            content = delta.get('content', '')
            token_text = ''
            if reasoning:
                if reasoning_token_count == 0:
                    token_text = '### Reasoning:\n'
                token_text += reasoning
                reasoning_token_count += 1
            if content and content != '':
                if content_token_count == 0:
                    token_text = '\n### Content:\n'
                token_text += content
                content_token_count += 1
            if token_text:
                full_response += token_text

            # Calculate tokens per second
            elapsed_time = time.time() - start_time
            tps = (reasoning_token_count + content_token_count) / elapsed_time if elapsed_time > 0 else 0

            # Clear the output and reprint everything
            clear_output(wait=True)
            display(Markdown(full_response))

    print(f"\nTokens per Second: {tps:.2f}\n")

In [49]:
inputs = "Solve this problem step by step: What is 15% of 240?"
output = stream_response(endpoint_name, inputs)

### Reasoning:
We are asked: "Solve this problem step by step: What is 15% of 240?" So we need to compute 15% of 240. Step by step: 15% means 15 per hundred, so it's 15/100 = 0.15. Multiply 0.15 * 240 = ? Alternatively, 10% of 240 is 24, 5% is 12, so 15% is 24+12=36. So answer is 36. Let's provide step-by-step solution
### Content:
To find 15% of 240, we can use the following steps:

1. Convert the percentage to a decimal: 15% = 15/100 = 0.15.
2. Multiply the decimal by the number: 0.15 × 240 = 36.

Alternatively, using fractions:
- 10% of 240 is 24 (since 240 ÷ 10 = 24).
- 5% of 240 is 12 (since 240 ÷ 20 = 12, or half of 10%).
- Then add them: 24 + 12 = 36.

Therefore, 15% of 240 is **36**.


Tokens per Second: 76.92



## Cleanup

In [50]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)